Train an RNN-based language model in Python to generate text from short input sequences.


In [ ]:
import numpy as np
import re

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

In [ ]:
corpus = """
king is a strong man
queen is a wise woman
boy is a young man
girl is a young woman
"""

sequence_length = 3
epochs = 200

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    return words

words = preprocess(corpus)

vocab = sorted(set(words))
word_to_index = {w: i+1 for i, w in enumerate(vocab)}  # +1 for padding safety
index_to_word = {i: w for w, i in word_to_index.items()}

vocab_size = len(vocab) + 1

encoded_words = [word_to_index[w] for w in words]

print("Encoded words:", encoded_words[:10])

Encoded words: [5, 4, 1, 8, 6, 7, 4, 1, 9, 10]


In [ ]:
X = []
y = []

for i in range(len(encoded_words) - sequence_length):
    input_seq = encoded_words[i:i+sequence_length]
    output_word = encoded_words[i+sequence_length]

    X.append(input_seq)
    y.append(output_word)

X = np.array(X)
y = np.array(y)

y = to_categorical(y, num_classes=vocab_size)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (17, 3)
y shape: (17, 12)


In [ ]:
model = Sequential()

model.add(Embedding(input_dim=vocab_size, output_dim=8, input_length=sequence_length))
model.add(SimpleRNN(32))
model.add(Dense(vocab_size, activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam')

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.fit(X, y, epochs=epochs, verbose=1)

Epoch 1/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - loss: 2.4879
Epoch 2/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 2.4803
Epoch 3/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 2.4728
Epoch 4/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 2.4653
Epoch 5/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 2.4578
Epoch 6/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 2.4502
Epoch 7/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 2.4425
Epoch 8/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 2.4346
Epoch 9/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 2.4266
Epoch 10/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 2.4183
Epoch 11/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 2.4099
Epoch 12/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 2.4011
Epoch 13/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 2.3920
Epoch 14/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 2.3827
Epoch 15/200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 2.3729
Epoch 16/200
1/1 ━━━━

In [ ]:
def generate_text(seed_text, next_words):
    for _ in range(next_words):

        # Preprocess seed
        seed = seed_text.lower().split()
        encoded = [word_to_index.get(w, 0) for w in seed]

        # Pad if needed
        encoded = pad_sequences([encoded], maxlen=sequence_length, truncating='pre')

        # Predict next word
        predicted = model.predict(encoded, verbose=0)
        predicted_word_index = np.argmax(predicted)

        predicted_word = index_to_word.get(predicted_word_index, "")

        seed_text += " " + predicted_word

    return seed_text

In [ ]:
print(generate_text("king is a", 5))
print(generate_text("queen is a", 5))

king is a strong man queen is a
queen is a wise woman boy is a
